# Loan Dataset

In this notebook, the loan dataset will be read in, prepared, analyzed and predicted. 

## 1.1 Notebook Preperation

To prepare the notebook to conduct these tasks, first we will import the necessary libraries, configure logging and read in the required dataset.

In [1]:
# Import libraries for notebook cell execution
import pandas as pd
import numpy as np 
import plotly.express as px
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

import logging
import sys

In [2]:
# Create logger instance
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s - %(message)s",
    datefmt="%H:%M:%S",
)

logger = logging.getLogger(__name__)


In [3]:
df_loan = pd.read_csv("../data/loan-10k.lrn.csv")
df_loan_test = pd.read_csv("../data/loan-10k.tes.csv")

## 1.2 Data Analysis

In [4]:
# Display dataset information:
df_loan.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 92 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ID                          10000 non-null  int64  
 1   loan_amnt                   10000 non-null  float64
 2   funded_amnt                 10000 non-null  float64
 3   funded_amnt_inv             10000 non-null  float64
 4   term                        10000 non-null  object 
 5   int_rate                    10000 non-null  float64
 6   installment                 10000 non-null  float64
 7   emp_length                  10000 non-null  object 
 8   home_ownership              10000 non-null  object 
 9   annual_inc                  10000 non-null  float64
 10  verification_status         10000 non-null  object 
 11  loan_status                 10000 non-null  object 
 12  pymnt_plan                  10000 non-null  object 
 13  purpose                     1000

In [5]:
# Display first rows of the dataframe:
display(df_loan.head())

,ID,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,emp_length,home_ownership,annual_inc,...,debt_settlement_flag,issue_d_month,issue_d_year,earliest_cr_line_month,earliest_cr_line_year,last_pymnt_d_month,last_pymnt_d_year,last_credit_pull_d_month,last_credit_pull_d_year,grade
0,24341,12500.0,12500.0,12500.0,36 months,7.21,387.17,< 1 year,MORTGAGE,81000.0,...,N,6,2018,6,2000,2,2019,2,2019,A
1,67534,33850.0,33850.0,33775.0,60 months,20.99,915.57,1 year,MORTGAGE,80000.0,...,N,10,2015,9,1984,2,2019,2,2019,E
2,35080,10000.0,10000.0,10000.0,60 months,20.00,264.94,< 1 year,RENT,36580.0,...,N,9,2017,10,2006,1,2018,11,2018,D
3,4828,20250.0,20250.0,20250.0,36 months,14.31,695.15,9 years,RENT,48700.0,...,N,0,2015,6,1996,6,2016,9,2017,C
4,59259,25000.0,25000.0,25000.0,36 months,14.99,866.52,1 year,MORTGAGE,85000.0,...,N,11,2016,0,2002,2,2019,2,2019,C


In [6]:
# Datset Dimensions
logger.info(f"Rows: {df_loan.shape[0]} Columns: {df_loan.shape[1]}")

15:28:17 [INFO] __main__ - Rows: 10000 Columns: 92


In [7]:
# Retrieve missing values
logger.info("Missing Values Per Column:")
display(df_loan.isnull().sum())
logger.info(f"Total missing values: {df_loan.isnull().sum().sum()}")

15:28:17 [INFO] __main__ - Missing Values Per Column:


ID                          0
loan_amnt                   0
funded_amnt                 0
funded_amnt_inv             0
term                        0
                           ..
last_pymnt_d_month          0
last_pymnt_d_year           0
last_credit_pull_d_month    0
last_credit_pull_d_year     0
grade                       0
Length: 92, dtype: int64

15:28:17 [INFO] __main__ - Total missing values: 0


There are no missing values that need to be taken care of.

In [8]:
# Value distribution of target class: grade
logger.info("Target Variable: Grade")
df_class_loan = df_loan["grade"].value_counts()
logger.info(df_class_loan.head(20))
logger.info("Count of distinct Review Authors: " + str(df_class_loan.size))

15:28:17 [INFO] __main__ - Target Variable: Grade
15:28:17 [INFO] __main__ - grade
C    2989
B    2881
A    1821
D    1449
E     621
F     182
G      57
Name: count, dtype: int64
15:28:17 [INFO] __main__ - Count of distinct Review Authors: 7


In [9]:
# Plot distribution of target class: grade
fig_grade = px.histogram(
    df_loan,
    x="grade",
    color="grade",
    category_orders={"grade": sorted(df_loan["grade"].unique())},
    text_auto=True
)
fig_grade.show()

When looking at the plot, we can see at the first glance the large imbalance of the target class grade. The grades C,B are the most common grades, while grade E,F and G are hugely underrepresent in the dataset. For even better clarity, we output the percentage distribution. 

In [10]:
# Numeric summary
grade_counts = df_loan["grade"].value_counts().sort_index()
grade_probability = df_loan["grade"].value_counts(normalize=True).sort_index()

balance_df = pd.DataFrame({
    "count": grade_counts,
    "proportion": grade_probability
})

logger.info(balance_df.sort_values(by="proportion", ascending=False))

15:28:18 [INFO] __main__ -        count  proportion
grade                   
C       2989      0.2989
B       2881      0.2881
A       1821      0.1821
D       1449      0.1449
E        621      0.0621
F        182      0.0182
G         57      0.0057


In [11]:
# Check for non-numeric features 
non_numeric_cols = df_loan.select_dtypes(exclude=['number']).columns
logging.info(non_numeric_cols)
num_non_numeric = len(non_numeric_cols)
logging.info(f"Number of non-numeric columns: {num_non_numeric}")

15:28:18 [INFO] root - Index(['term', 'emp_length', 'home_ownership', 'verification_status',
       'loan_status', 'pymnt_plan', 'purpose', 'addr_state',
       'initial_list_status', 'application_type', 'hardship_flag',
       'disbursement_method', 'debt_settlement_flag', 'grade'],
      dtype='object')
15:28:18 [INFO] root - Number of non-numeric columns: 14


There are 14 non-numeric columns that need potential encoding to numeric columns. 

For an accurate handling of outliers we first have to encode the non-numerical values in order to perform z-score evaluation. Therefore it will be moved to data preprocessing. 

## 1.3 Data Preprocessing 

No missing values have to be handled, therefore this step will be skipped. 
Now we will handle to encode the non-numeric values to be better processable by the used models. 
We first have to identify if the data is ordinal or nominal categorical values in order to know to encode it. 
Therefore, we will investigate the 14 non-numerical columns of the dataset. 

In [12]:
# Display df of non-numeric values
display(df_loan[non_numeric_cols])
# Display unique values 
for column in non_numeric_cols:
    logger.info(f"Unique values for columns: {column} {df_loan[column].unique()}")

,term,emp_length,home_ownership,verification_status,loan_status,pymnt_plan,purpose,addr_state,initial_list_status,application_type,hardship_flag,disbursement_method,debt_settlement_flag,grade
0,36 months,< 1 year,MORTGAGE,Not Verified,Current,n,debt_consolidation,IL,w,Individual,N,Cash,N,A
1,60 months,1 year,MORTGAGE,Source Verified,Current,n,debt_consolidation,OK,f,Individual,N,Cash,N,E
2,60 months,< 1 year,RENT,Not Verified,Fully Paid,n,debt_consolidation,AZ,w,Individual,N,Cash,N,D
3,36 months,9 years,RENT,Source Verified,Charged Off,n,debt_consolidation,CA,f,Individual,N,Cash,N,C
4,36 months,1 year,MORTGAGE,Source Verified,Current,n,debt_consolidation,FL,w,Individual,N,Cash,N,C
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,60 months,5 years,OWN,Verified,Current,n,debt_consolidation,NJ,w,Individual,N,Cash,N,C
9996,36 months,10+ years,RENT,Source Verified,Charged Off,n,credit_card,NY,w,Individual,N,Cash,N,B
9997,60 months,< 1 year,RENT,Source Verified,Current,n,debt_consolidation,OR,w,Individual,N,Cash,N,C
9998,60 months,1 year,RENT,Source Verified,Charged Off,n,medical,CA,w,Individual,N,Cash,N,D


15:28:18 [INFO] __main__ - Unique values for columns: term [' 36 months' ' 60 months']
15:28:18 [INFO] __main__ - Unique values for columns: emp_length ['< 1 year' '1 year' '9 years' '10+ years' '3 years' '4 years' '7 years'
 '2 years' '5 years' '6 years' '8 years']
15:28:18 [INFO] __main__ - Unique values for columns: home_ownership ['MORTGAGE' 'RENT' 'OWN' 'ANY' 'OTHER']
15:28:18 [INFO] __main__ - Unique values for columns: verification_status ['Not Verified' 'Source Verified' 'Verified']
15:28:18 [INFO] __main__ - Unique values for columns: loan_status ['Current' 'Fully Paid' 'Charged Off' 'Late (31-120 days)'
 'In Grace Period' 'Late (16-30 days)']
15:28:18 [INFO] __main__ - Unique values for columns: pymnt_plan ['n' 'y']
15:28:18 [INFO] __main__ - Unique values for columns: purpose ['debt_consolidation' 'car' 'credit_card' 'other' 'major_purchase'
 'home_improvement' 'small_business' 'medical' 'vacation' 'moving' 'house'
 'renewable_energy' 'wedding']
15:28:18 [INFO] __main__ - Un

From these unique values we can gather the type of non-numerical values and therefore the encoding techniques:
term -> direct cast to month duration. As we also have svm which responds 
emp year: cast to years, <1 year will be cast to 0.5, 10+ to 10
home ownership will become one hot 
verification_status will become ordinal as it can be ranked
loan_status is also ranked so therefore we can rank the statuses from good to worse 
payment plan will become binary 0 1
purpose categorical also become one hot 
addr_state non categorical but too many adresses 
initial_list_status binary 0 1
application type binary 0 1
hardship_flag binary 0 1
disbursement_method binary 0 1 
debt_settlement_flag binary 0 1
grade ordinal encoding as grades have ranking 

In [13]:
encoded_loan_df = df_loan.copy()

In [14]:
# Encode binary columns
binary_columns = {"pymnt_plan", "initial_list_status", "application_type", "hardship_flag", "disbursement_method", "debt_settlement_flag"}
le = LabelEncoder()
for column in binary_columns:
    encoded_loan_df[column] = le.fit_transform(df_loan[column])
    logger.info(f"Unique label values of col: {column}: {encoded_loan_df[column].unique()}")
# Verify binary columns encoding 
display(encoded_loan_df[list(binary_columns)])

15:28:18 [INFO] __main__ - Unique label values of col: hardship_flag: [0 1]
15:28:18 [INFO] __main__ - Unique label values of col: pymnt_plan: [0 1]
15:28:18 [INFO] __main__ - Unique label values of col: debt_settlement_flag: [0 1]
15:28:18 [INFO] __main__ - Unique label values of col: initial_list_status: [1 0]
15:28:18 [INFO] __main__ - Unique label values of col: disbursement_method: [0 1]
15:28:18 [INFO] __main__ - Unique label values of col: application_type: [0 1]


,hardship_flag,pymnt_plan,debt_settlement_flag,initial_list_status,disbursement_method,application_type
0,0,0,0,1,0,0
1,0,0,0,0,0,0
2,0,0,0,1,0,0
3,0,0,0,0,0,0
4,0,0,0,1,0,0
...,...,...,...,...,...,...
9995,0,0,0,1,0,0
9996,0,0,0,1,0,0
9997,0,0,0,1,0,0
9998,0,0,0,1,0,0


In [15]:
# Encode ordinal values to ranked 
ordinal_mapping = {
    "grade" : {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4, "F": 6, "G": 7},
    "verification_status" : {"Verified": 0, "Source Verified": 1, "Not Verified": 2},
    "loan_status" : {"Fully Paid": 0, "Current": 1, "In Grace Period": 2, "Late (16-30 days)":3, "Late (31-120 days)": 4, "Charged Off": 5 },
}
encoded_loan_df = encoded_loan_df.replace(ordinal_mapping)
# Verify mapping
unique_per_column = {col: encoded_loan_df[col].unique() for col in ordinal_mapping.keys()}
logging.info(unique_per_column)

/var/folders/ld/yxrhvq2x3zzcf_rdhkny7drw0000gp/T/ipykernel_9184/449838390.py:7: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

15:28:18 [INFO] root - {'grade': array([0, 4, 3, 2, 1, 7, 6]), 'verification_status': array([2, 1, 0]), 'loan_status': array([1, 0, 5, 4, 2, 3])}


In [16]:
# Encode direct numerical mapping 
# For the term column just extract the number 
encoded_loan_df['term'] = df_loan['term'].str.extract(r'(\d+)').astype(int)
logging.info(encoded_loan_df['term'].unique())

# years helper function 
def convert_years(x):
    x = x.strip()
    if '< 1' in x:
        return 0.5
    elif '10+' in x:
        return 10
    else:
        num = pd.to_numeric(''.join(filter(str.isdigit, x)))
        return num

encoded_loan_df['emp_length'] = df_loan['emp_length'].apply(convert_years)
logging.info(encoded_loan_df['emp_length'].unique())

15:28:18 [INFO] root - [36 60]
15:28:18 [INFO] root - [ 0.5  1.   9.  10.   3.   4.   7.   2.   5.   6.   8. ]


In [17]:
# One Hot Encoding
one_hot_columns = ["home_ownership", "purpose", "addr_state"]

for column in one_hot_columns:
    enc = OneHotEncoder(drop=None, sparse_output=False)  # modern syntax
    transformed = enc.fit_transform(encoded_loan_df[[column]])
    col_names = [f"{column}_{cat}" for cat in enc.categories_[0]]
    df_enc = pd.DataFrame(transformed, columns=col_names, index=encoded_loan_df.index)
    encoded_loan_df = pd.concat([encoded_loan_df.drop(columns=[column]), df_enc], axis=1)

In [18]:
logger.info(encoded_loan_df.shape[1])

15:28:18 [INFO] __main__ - 157
